In [1]:
import pandas as pd
from os import listdir

pd.set_option("display.precision", 2)
psic_files = listdir('results/psic/graded/')
models = [file.strip('.csv')[12:] for file in psic_files]
models = [model if model[0] != 'o' else model[7:] for model in models]
print(models)
psic_dict = {name: pd.read_csv('results/psic/graded/'+file) for (name,file) in zip(models, psic_files)}

['claude', 'r1', 'claude-no-thinking', 'gpt-5-high', 'grok-3-mini', 'gpt-5-high']


# Showcasing how the results look like
- each model has a csv file in the same style, bellow you can see the deepseek reasoner model as an example.
- The 'Type' column shows an abraviation of the type of the test (i.e. SS_2pretend are the strange stories involving pretend play).
- The 'S' column shows the system-level instruction given to the model.
- The 'Q_1', 'Q_2', 'A_1', and 'A_2' show, respectively, the first and second questions asked to and answers given by the model.
- The 'CA_1' and 'CA_2' are the correct answers.
- The 'C' is the evaluation if the model answered to the question correctly. An answer is only accepted if the reasoning is deemed correct.

In [2]:
psic_dict['claude'].iloc[[0]]

,Type,S,Q_1,CA_1,A_1,R_1,Q_2,CA_2,A_2,R_2,Level,Q_type,Deviation,C
0,SA_1fb,You will be asked a question. Please respond t...,Sally and Anne are playing. Sally has a box an...,basket,The ball is in Anne's basket.,This is a straightforward factual question abo...,Where does Sally look for the ball?,box,Sally looks for the ball in her box.,This is a classic Theory of Mind test called t...,NaN,NaN,0.0,2


# Assessing the performance of each model in each particular type of task
- First, I simply just check how many times they get it right and wrong to access accuracy

In [3]:
import numpy as np

types = psic_dict['claude']['Type'].unique()
accuracy = {model: {} for model in models}
for (i,model) in zip(range(len(models)),models):
    for typ in types:
        accuracy[model][typ] = np.sum((psic_dict[model]['Type'] == typ) * psic_dict[model]['C'])\
        /(np.sum(psic_dict[model]['Type'] == typ)*2)
    accuracy[model]['overall'] = np.sum(psic_dict[model]['C'])/(2*len(psic_dict[model]))

In [4]:
accuracy = pd.DataFrame.from_dict(accuracy)
accuracy

,claude,r1,claude-no-thinking,gpt-5-high,grok-3-mini
SA_1fb,1.00,1.00,1.00,1.00,1.00
SA_2fb,1.00,1.00,1.00,1.00,0.67
SS_1lie,1.00,1.00,1.00,1.00,1.00
SS_2pretend,1.00,1.00,1.00,1.00,1.00
SS_3joke,1.00,1.00,1.00,1.00,1.00
SS_4whitelie,1.00,1.00,1.00,1.00,1.00
SS_5misunderstanding,1.00,1.00,1.00,1.00,1.00
SS_6sarcasm,0.83,1.00,1.00,1.00,0.83
SS7_dubblebluff,0.83,0.83,0.67,1.00,1.00
H_1,1.00,0.94,0.88,1.00,1.00


In [5]:
SS = types[2:-2]
print(accuracy.loc[SS].T.style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
        .set_properties(**{'font-size': '20px'}).format(precision=2).to_latex())

\begin{tabular}{lrrrrrrr}
 & SS_1lie & SS_2pretend & SS_3joke & SS_4whitelie & SS_5misunderstanding & SS_6sarcasm & SS7_dubblebluff \\
claude & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fbb61a \color#000000 \font-size20px 0.83 & \background-color#fbb61a \color#000000 \font-size20px 0.83 \\
r1 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fbb61a \color#000000 \font-size20px 0.8

In [6]:
SA = types[:2]
print(accuracy.loc[SA].rename({'SA_1fb':'SA-1','SA_2fb':'SA-2'}, axis='index').T.style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
         .set_properties(**{'font-size': '20px'}).format(precision=2).to_latex())

\begin{tabular}{lrr}
 & SA-1 & SA-2 \\
claude & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
r1 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
claude-no-thinking & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
gpt-5-high & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
grok-3-mini & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#ed6925 \color#f1f1f1 \font-size20px 0.67 \\
\end{tabular}



In [7]:
IM = psic_dict['claude'][(psic_dict['claude']['Type'] == 'H_1') | (psic_dict['claude']['Type'] == 'H_3')]
levels = IM['Level'].unique()
IM_intentional = {model: {} for model in models}
IM_memory = {model: {} for model in models}
for (i,model) in zip(range(len(models)),models):
    for level in levels:
        IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
                                               (psic_dict[model]['Q_type'] == 'I')) * psic_dict[model]['C'])\
                                                /(np.sum((psic_dict[model]['Level'] == level) & (psic_dict[model]['Q_type'] == 'I'))*2)
        IM_memory[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
                                          (psic_dict[model]['Q_type'] == 'M')) * psic_dict[model]['C'])\
                                        /(np.sum((psic_dict[model]['Level'] == level) & (psic_dict[model]['Q_type'] == 'M'))*2)

    # accuracy[model]['overall'] = np.sum(psic_dict[model]['C'])/(2*len(psic_dict[model]))
intentional_df = pd.DataFrame.from_dict(IM_intentional)
memory_df = pd.DataFrame.from_dict(IM_memory)

/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_intentional[model][level] = np.sum(((psic_dict[model]['Level'] == level) &\
/tmp/ipykernel_411402/3025178632.py:7: RuntimeWarning: invalid value encountered in scalar divide
  IM_in

In [8]:
intentional_df.index = memory_df.index.astype(int)
print(intentional_df.iloc[1:].T.style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
        .set_properties(**{'font-size': '20px'}).format(precision=2).to_latex())

\begin{tabular}{lrrrr}
 & 2 & 3 & 4 & 5 \\
claude & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
r1 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#bc3754 \color#f1f1f1 \font-size20px 0.50 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
claude-no-thinking & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#bc3754 \color#f1f1f1 \font-size20px 0.50 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
gpt-5-high & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#f98e09 \color#f1f1f1 \font-size20px 0.75 & \background-color#fcffa4 \color#000000 \font-size20p

In [9]:
memory_df.index = memory_df.index.astype(int)
print(memory_df.T.style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
        .set_properties(**{'font-size': '20px'}).format(precision=2).to_latex())

\begin{tabular}{lrrrrr}
 & 1 & 2 & 3 & 4 & 5 \\
claude & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
r1 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#f98e09 \color#f1f1f1 \font-size20px 0.75 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\
claude-no-thinking & \background-color#f9cb35 \color#000000 \font-size20px 0.88 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 & \background-color#f9cb35 \color#000000 \font-size20px 0.88 & \background-color#fcffa4 \color#000000 \font-size20px 1.00 \\

In [10]:
((memory_df+intentional_df)/2).style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
        .set_properties(**{'font-size': '20px'}).format(precision=2)

,claude,r1,claude-no-thinking,gpt-5-high,grok-3-mini
1,nan,nan,nan,nan,nan
2,1.00,1.00,1.00,1.00,1.00
3,1.00,0.75,0.75,0.88,0.88
4,1.00,0.88,0.94,1.00,0.88
5,1.00,1.00,1.00,1.00,1.00


## Modifications

In [11]:
mod_files = listdir('results/mod/graded/')
models = [file.strip('.csv')[11:] for file in mod_files]
mod_dict = {name: pd.read_csv('results/mod/graded/'+file, index_col=0) for (name,file) in zip(models, mod_files)}
#little cosmetic mod so the name of no reas-claude is not giant
mod_dict['no-reas-claude'] = mod_dict['no-reas-claude-no-thinking']
del mod_dict['no-reas-claude-no-thinking']
models = mod_dict.keys()

In [12]:
mod_dict['no-reas-claude']

,Q_1,CA_1,A_1,Q_2,A_2,C
Unnamed: 0,,,,,,
1A.1,"There is a shoe box, inside of it there are ki...",FALSE,True.\n\nJake would likely be delighted to fin...,Why?,You're right to question my reasoning. Let me ...,1
1A.2,"There is a shoe box, inside of it there are ki...",Kids toys,Jake would likely believe the box contains Nik...,Why?,You're right to question my reasoning. Actuall...,1
1B.1,"Jake is on the road, he wants to find his way ...",FALSE,"False.\n\nIf Jake cannot read, he would not be...",Why?,Jake would not be relieved because:\n\n1. **He...,2
1B.2,"Jake is on the road, he wants to find his way ...","Not enough info, he doesn’t understand the sign",This is a bit of a trick question! \n\nEven th...,Why?,You're right to question my overthinking! \n\n...,1
1C.1,"There is a shoe box, inside of it there are ki...",FALSE,False.\n\nJake would not be delighted to find ...,Why?,Jake would not be delighted because:\n\n1. **H...,2
1C.2,"There is a shoe box, inside of it there are ki...",Kids toys,Jake believes that the box contains **kids toy...,Why?,Jake believes the box contains kids toys becau...,2
1D.1,"There is a shoe box, Jake puts kids’ toys insi...",FALSE,True.\n\nEven though Jake put kids' toys in th...,Why?,You're right to question my reasoning. Let me ...,1
1D.2,"There is a shoe box, Jake puts kids’ toys insi...",Kids toys,Jake believes that the box contains Nike shoes...,Why?,You're right to question this! Let me reconsid...,1
2A.1,"In the room there are Adu, Lin, a flower, an a...",Aquarium,Adu will look for the flower in the **bag** (t...,Why?,"This is a classic ""false belief"" scenario that...",0


In [13]:
idxs = mod_dict['claude'].index
accuracy = {model: {} for model in models}
for (i,model) in zip(range(len(models)),models):
    for idx in idxs:
        accuracy[model][idx] = mod_dict[model].loc[idx]['C']/2
    # accuracy[model]['overall'] = np.sum(psic_dict[model]['C'])/(2*len(psic_dict[model]))
pd.DataFrame.from_dict(accuracy).T.style.background_gradient(cmap ='inferno', vmin=0, vmax=1)\
        .set_properties(**{'font-size': '15px'}).format(precision=2)

,1A.1,1A.2,1B.1,1B.2,1C.1,1C.2,1D.1,1D.2,2A.1,2A.2,2B.1,2B.2,2C.1,2C.2
claude,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00
grok-3-mini,1.00,0.50,1.00,1.00,1.00,1.00,1.00,0.50,0.00,0.00,0.00,0.00,1.00,1.00
gpt-5-high,1.00,1.00,1.00,0.00,1.00,1.00,1.00,0.50,0.50,0.50,0.00,0.00,1.00,1.00
r1,1.00,1.00,1.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00
no-reas-claude,0.50,0.50,1.00,0.50,1.00,1.00,0.50,0.50,0.00,0.00,0.00,0.00,0.50,1.00


# We now check what is wrong with particular prompts

In [14]:
from tabulate import tabulate
def print_result(prompt):
    print(f'''
{'\n\n-------\n\n'.join(map(str, list(prompt.Type)))}

- System:
{'\n\n-------\n\n'.join(map(str, list(prompt.S)))}

- Question 1:
{'\n\n-------\n\n'.join(map(str, list(prompt.Q_1)))}

- Reasoning 1:
{'\n\n-------\n\n'.join(map(str, list(prompt.R_1)))}

- Answer 1:
{'\n\n-------\n\n'.join(map(str, list(prompt.A_1)))}

- Correct Answer 1:
{'\n\n-------\n\n'.join(map(str, list(prompt.CA_1)))}

- Question 2:
{'\n\n-------\n\n'.join(map(str, list(prompt.Q_2)))}

- Reasoning 2:
{'\n\n-------\n\n'.join(map(str, list(prompt.R_2)))}

- Answer 2:
{'\n\n-------\n\n'.join(map(str, list(prompt.A_2)))}

- Correct Answer 2:
{'\n\n-------\n\n'.join(map(str, list(prompt.CA_2)))}''')
    

## Strange stories
### Sarcasm

In [71]:
prompt = psic_dict['claude'][(psic_dict['claude']['Type'] == 'SS_6sarcasm') & (psic_dict['claude']['C'] == 1)]
print_result(prompt)


SS_6sarcasm

- System:
You will be asked a question. Please respond to it as accurately as possible without using many words.

- Question 1:
Kim promised her father to help clean the kitchen. She first cleaned the sink and then all the doors of the cabinets. She thought it would be a good idea to also clean the cabinets from the inside. When she opened one of them, a pack flour fell out of it on the ground. It exploded and flour was everywhere. At that point, Kim's father entered the kitchen. He said: "Wow, everything is so clean now!" 
Is it true what Kim's father says?

- Reasoning 1:
Let me think about this situation step by step:

1. Kim cleaned the sink and cabinet doors
2. She then opened a cabinet to clean inside
3. A pack of flour fell and exploded, spreading flour everywhere
4. At that point, her father entered and said everything is clean

The father's statement that "everything is so clean now" is not accurate because flour just exploded and is everywhere in the kitchen. Th

In [16]:
prompt = psic_dict['grok-3-mini'][(psic_dict['grok-3-mini']['Type'] == 'SS_6sarcasm') & (psic_dict['grok-3-mini']['C'] == 1)]
print_result(prompt)


SS_6sarcasm

- System:
You will be asked a question. Please respond to it as accurately as possible without using many words.

- Question 1:
Kim promised her father to help clean the kitchen. She first cleaned the sink and then all the doors of the cabinets. She thought it would be a good idea to also clean the cabinets from the inside. When she opened one of them, a pack flour fell out of it on the ground. It exploded and flour was everywhere. At that point, Kim's father entered the kitchen. He said: "Wow, everything is so clean now!" 
Is it true what Kim's father says?

- Reasoning 1:
First, the user asks: "Is it true what Kim's father says?" Kim's father said, "Wow, everything is so clean now!"

From the story:

- Kim cleaned the sink and the doors of the cabinets.

- She decided to clean the inside of the cabinets.

- When she opened one, a pack of flour fell and exploded, making flour everywhere.

- At that moment, her father enters and says it's clean.

So, is it true that every

## Imposing memories

In [58]:
prompts = psic_dict['grok-3-mini'][((psic_dict['grok-3-mini']['Type'] == 'H_1') | (psic_dict['grok-3-mini']['Type'] == 'H_3')) & (psic_dict['grok-3-mini']['C'] == 0)]
print_result(prompts.iloc[[1]])


H_3

- System:
You will be provided a story. At the end there is a yes/no question. Please answer as accurately as possible.

- Question 1:
Meet Hannah. She is late for the lunch break and when she comes in, all the places are taken. Then, she discovers there is a free chair next to Ama and her boyfriend, Matthias. Hannah: Hi guys, I'm late, can I sit here? Ama: Oh actually, I was saving that seat for Abi. Hannah: Oh, okay. Ama: Sorry! Hannah meets Carolyn. Hannah: Hey Carolyn, can I sit here? Carolyn: Sure. Carolyn: Hey are you OK? You look a bit upset. Hannah: Yeah I'm alright, it's just that Ama is so mean sometimes. Carolyn: What happened? Hannah: Well, I came in to lunch a bit late, and there were hardly any seats left, but when I asked if I could sit at the table with her and Matthias and their friends, Ama said she was saving the seat for Abi! Carolyn: So? Ama and Abi always save seats for each other. Hannah: But Abi went home sick this morning! So Ama couldn't have been saving

### Double bluff

In [70]:
# prompt = psic_dict['grok-3-mini'][(psic_dict['grok-3-mini']['Type'] == 'SS6_sarcasm')]# & (psic_dict['claude']['C'] == 1)]
# print_result(prompt)
list(psic_dict['grok-3-mini']['Type'].items())

[(0, 'SA_1fb'),
 (1, 'SA_1fb'),
 (2, 'SA_1fb'),
 (3, 'SA_2fb'),
 (4, 'SA_2fb'),
 (5, 'SA_2fb'),
 (6, 'SS_1lie'),
 (7, 'SS_1lie'),
 (8, 'SS_1lie'),
 (9, 'SS_2pretend'),
 (10, 'SS_2pretend'),
 (11, 'SS_2pretend'),
 (12, 'SS_3joke'),
 (13, 'SS_3joke'),
 (14, 'SS_3joke'),
 (15, 'SS_4whitelie'),
 (16, 'SS_4whitelie'),
 (17, 'SS_4whitelie'),
 (18, 'SS_5misunderstanding'),
 (19, 'SS_5misunderstanding'),
 (20, 'SS_5misunderstanding'),
 (21, 'SS_6sarcasm'),
 (22, 'SS_6sarcasm'),
 (23, 'SS_6sarcasm'),
 (24, 'SS7_dubblebluff'),
 (25, 'SS7_dubblebluff'),
 (26, 'SS7_dubblebluff'),
 (27, 'H_1'),
 (28, 'H_1'),
 (29, 'H_1'),
 (30, 'H_1'),
 (31, 'H_1'),
 (32, 'H_1'),
 (33, 'H_1'),
 (34, 'H_1'),
 (35, 'H_1'),
 (36, 'H_1'),
 (37, 'H_1'),
 (38, 'H_1'),
 (39, 'H_1'),
 (40, 'H_1'),
 (41, 'H_1'),
 (42, 'H_1'),
 (43, 'H_3'),
 (44, 'H_3'),
 (45, 'H_3'),
 (46, 'H_3'),
 (47, 'H_3'),
 (48, 'H_3'),
 (49, 'H_3'),
 (50, 'H_3'),
 (51, 'H_3'),
 (52, 'H_3'),
 (53, 'H_3'),
 (54, 'H_3'),
 (55, 'H_3'),
 (56, 'H_3'),
 (57,

In [18]:
prompt = psic_dict['r1'][(psic_dict['r1']['Type'] == 'SS7_dubblebluff') & (psic_dict['r1']['C'] == 1)]
print_result(prompt)


SS7_dubblebluff

- System:
You will be asked a question. Please respond to it as accurately as possible without using many words.

- Question 1:
Jip plays hide and seek with his friends. He has to quickly choose where to hide, behind the tree or the shed. If Jip is found immediately, he has lost. You can hide better behind the shed, so that's the best place. But whoever has to look for him probably thinks so too. Jip is very smart: he hides behind the shed after all!
Where does Jip hide?

- Reasoning 1:
First, the user is describing a scenario where Jip is playing hide and seek. He has two choices: hide behind the tree or behind the shed. The goal is not to be found immediately, so if he's found, he loses.

It says that hiding behind the shed is better because you can hide better there. But the seeker might also think that, so they might check the shed first because it's the obvious good hiding spot.

Jip is described as very smart, and he hides behind the shed after all. The question

In [19]:
prompt = psic_dict['gpt-5-high'][(psic_dict['gpt-5-high']['Type'] == 'SS7_dubblebluff') & (psic_dict['r1']['C'] == 1)]
print_result(prompt)

AttributeError: 'DataFrame' object has no attribute 'R_1'

With this particular analysis we learn that in both cases the model couldn't make full sense of the question: in the first it answers that the assumption of hannah was wrong, when in fact it had to respond if that was her assumption, while in the second it doesn't understand that Carolyn knowing something was wrong refered to Hannah and not to Abi.

## Strategies

In [ ]:
print_result(psic_dict['claude'][psic_dict['claude'].index == 55])

In [ ]:
print_result(psic_dict['claude'][psic_dict['claude'].index == 58])

## Non reasoning results

In [76]:
def print_mod_no_reas(name,idx):
    for attribute in mod_dict[name].iloc[[idx]]:
        print(f'{attribute}:{mod_dict[name].iloc[[idx]][attribute].item()}\n')

In [113]:
print_mod_no_reas('gpt-5-high',8)

Q:In the room there are Adu, Lin, a flower, an aquarium, and a transparent plastic bag with water. Adu takes the flower and puts it in the bag. He leaves the room and goes to school. While Adu is away, Lin takes the flower out of the bag and puts it in the aquarium. Lin leaves the room and goes to his swimming class. Adu comes back from school and enters the room. He doesn’t know what happened in the room when he was away. Adu will look for the flower in the …


CA:Aquarium

A:plastic bag (where he left it)

R:**Interpreting theory of mind**

I'm looking at this as a theory of mind or false-belief question, similar to the Sally-Anne test. In this scenario, we have Adu, Lin, a flower, and a few objects. Adu places the flower in a transparent bag and leaves the room. While he's away, Lin moves the flower to the aquarium.

The twist here is that the bag is transparent, which may mean Adu can see the flower in the aquarium. I need to figure out if that changes his expectations.

**Explorin

In [102]:
print_mod_no_reas('no-reas-claude',3)

Q_1:Jake is on the road, he wants to find his way to Leiden but doesn't know the directions. He sees a sign that points out that Amsterdam is to the right and Leiden is to the left. Jake does not know how to read. To which direction Jake will turn his car?

CA_1:Not enough info, he doesn’t understand the sign

A_1:This is a bit of a trick question! 

Even though Jake cannot read the words "Amsterdam" and "Leiden" on the sign, he would still need some way to know he wants to go to Leiden. If he truly cannot read at all, he wouldn't be able to recognize the written word "Leiden" as his destination.

However, if we assume Jake knows he needs to go to Leiden through some other means (perhaps someone told him verbally, or he has it marked on a map with a symbol), he still wouldn't be able to match that knowledge to the text on the sign since he can't read.

Therefore, **Jake would not be able to determine which direction to turn** based on the sign alone. He would need additional help, such

In [22]:
print_result(psic_dict['grok-3-mini'][psic_dict['grok-3-mini'].index == 5])


SA_2fb

- System:
You will be asked a question. Please respond to it as accurately as possible without using many words.

- Question 1:
Tom and Gerald went swimming. But when they wanted to go home, Tom found out that his bike has a flat tire. When they start walking they see the swimming teacher passing in his car. They put up their hand and the swimming teacher stops the car at the bus stop alongside the road. They ask whether they can hitchhike, the swimming teacher says yes. 
Tom says he forgot his bag in the dressing room, so he quickly runs to grab it. The swimming teacher says to Gerald that he'll park his car in the parking lot around the corner, so they can put the bikes in the trunk.
On the way to the parking lot the swimming teacher sees Tom and tells him where he is going. So when Tom has his bag, he walks to the parking lot. 
Gerald walks to the dressing room and asks where Tom is. The children in the dressing room say that Tom went to the car of the swimming teacher. 
Wh

In [83]:
print(mod_dict['r1'].loc['1C.1'].R)

First, the scenario: There's a shoe box with kids' toys inside, but you can't see inside. The label says "nike shoes." Jake's friend told him that there are kids' toys inside and to ignore the label. Jake believed his friend. Then Jake finds the box, reads the label, and gets delighted because he's a shoe enthusiast.

The question is whether it's true or false that Jake is delighted.

Jake believed his friend, so he knows that there are kids' toys inside, not shoes. But when he sees the label, which says "nike shoes," he might forget or disregard what his friend said because he's a shoe enthusiast. However, the problem says that Jake believed his friend, so he should still believe that there are toys inside.

But the key point is that Jake is delighted when he reads the label. Since he believes his friend, he knows that the label is misleading, so he shouldn't be delighted by the label because he knows it's not true. But he is a shoe enthusiast, so the thought of shoes might excite him

In [101]:
psic_dict['grok-3-mini'][psic_dict['grok-3-mini'].Type == 'SS7_dubblebluff']

,Type,S,Q_1,CA_1,A_1,R_1,Q_2,CA_2,A_2,R_2,Level,Q_type,Deviation,C
24,SS7_dubblebluff,You will be asked a question. Please respond t...,"During the war, the Red Army kidnaps someone f...",True,No.,"First, the user asks: ""Is it true what the pri...",Why does the prisoner say this?,NaN,"He says it to deceive the Red Army, knowing th...","First, the user is asking why the prisoner say...",NaN,NaN,0.0,2
25,SS7_dubblebluff,You will be asked a question. Please respond t...,"During the war, the Roman Army kidnaps someone...",True,"No, because the prisoner is lying.","First, the user asks: ""Is it true what the pri...",Why does the prisoner say this?,NaN,Because he is lying to mislead the Roman Army ...,"First, the user is asking why the prisoner say...",NaN,NaN,1.0,2
26,SS7_dubblebluff,You will be asked a question. Please respond t...,Jip plays hide and seek with his friends. He h...,Shed,Behind the shed.,"First, the user asks: ""Where does Jip hide?"" b...",Jip realizes that the others will also think t...,NaN,"Because it's the best hiding spot, and Jip ant...","First, the user is asking why Jip hides behind...",NaN,NaN,2.0,2


In [110]:
mod_dict['claude'].iloc[10]

Q     In the room there are Adu, Lin, a flower, a su...
CA                                             Suitcase
A     Adu will look for the flower on the **chest**....
R     Let me trace through this step by step:\n\n1. ...
C                                                     0
Name: 2B.1, dtype: object